In [ ]:
# Storm Stats
import geopandas as gpd
import pandas as pd
import numpy as np
import glob
import os
import rasterio
from rasterstats import zonal_stats


shapefile = "../../public/shapefiles/county.shp"

In [32]:
import rasterio
import numpy as np

for date in np.arange(10, 16+1):
    print(f"Processing date: 2026-03-{date:02d} cumulative")
    tif_path = f"./storm_data/2026_03_{date:02d}_cumulative.tif"

    with rasterio.open(tif_path) as src:
        arr = src.read(1).astype(float)

        # Handle nodata
        if src.nodata is not None:
            arr = np.where(arr == src.nodata, np.nan, arr)

        # Compute stats (convert mm → inches here too)
        min_val = np.nanmin(arr) / 25.4
        mean_val = np.nanmean(arr) / 25.4
        max_val = np.nanmax(arr) / 25.4

        print({
            "min": round(min_val, 2),
            "mean": round(mean_val, 2),
            "max": round(max_val, 2)
        })

Processing date: 2026-03-10 cumulative
{'min': np.float64(0.0), 'mean': np.float64(0.2), 'max': np.float64(6.26)}
Processing date: 2026-03-11 cumulative
{'min': np.float64(0.01), 'mean': np.float64(0.96), 'max': np.float64(8.4)}
Processing date: 2026-03-12 cumulative
{'min': np.float64(0.06), 'mean': np.float64(1.81), 'max': np.float64(14.03)}
Processing date: 2026-03-13 cumulative
{'min': np.float64(0.26), 'mean': np.float64(4.74), 'max': np.float64(25.57)}
Processing date: 2026-03-14 cumulative
{'min': np.float64(1.45), 'mean': np.float64(11.36), 'max': np.float64(58.42)}
Processing date: 2026-03-15 cumulative
{'min': np.float64(1.76), 'mean': np.float64(12.45), 'max': np.float64(59.89)}
Processing date: 2026-03-16 cumulative
{'min': np.float64(1.81), 'mean': np.float64(13.04), 'max': np.float64(61.37)}


In [36]:
import rasterio
import numpy as np

for date in np.arange(10, 16+1):
    print(f"Processing date: 2026-03-{date:02d} daily")
    tif_path = f"./storm_data/2026_03_{date:02d}.tif"

    with rasterio.open(tif_path) as src:
        arr = src.read(1).astype(float)

        # Handle nodata
        if src.nodata is not None:
            arr = np.where(arr == src.nodata, np.nan, arr)

        # Compute stats (convert mm → inches here too)
        min_val = np.nanmin(arr) / 25.4
        mean_val = np.nanmean(arr) / 25.4
        max_val = np.nanmax(arr) / 25.4

        print({
            "min": round(min_val, 2),
            "mean": round(mean_val, 2),
            "max": round(max_val, 2)
        })

Processing date: 2026-03-10 daily
{'min': np.float64(0.0), 'mean': np.float64(0.2), 'max': np.float64(6.26)}
Processing date: 2026-03-11 daily
{'min': np.float64(0.0), 'mean': np.float64(0.76), 'max': np.float64(4.82)}
Processing date: 2026-03-12 daily
{'min': np.float64(0.02), 'mean': np.float64(0.85), 'max': np.float64(5.63)}
Processing date: 2026-03-13 daily
{'min': np.float64(0.04), 'mean': np.float64(2.94), 'max': np.float64(22.39)}
Processing date: 2026-03-14 daily
{'min': np.float64(0.0), 'mean': np.float64(6.62), 'max': np.float64(37.25)}
Processing date: 2026-03-15 daily
{'min': np.float64(0.0), 'mean': np.float64(1.08), 'max': np.float64(6.26)}
Processing date: 2026-03-16 daily
{'min': np.float64(0.0), 'mean': np.float64(0.59), 'max': np.float64(3.02)}


In [34]:
gdf = gpd.read_file(shapefile).dissolve(by="county", as_index=False).reset_index(drop=True)

print(gdf["county"].unique())  

for date in np.arange(10, 16+1):
  print(f"Processing date: 2026-03-{date:02d} cumulative")
  tif_path = f"./storm_data/2026_03_{date:02d}_cumulative.tif"

  for county in gdf["county"].unique():
      county_gdf = gdf[gdf["county"] == county]
      stats = zonal_stats(county_gdf, tif_path, stats=["mean", "min", "max"])

      # Round values
      rounded_stats = [
          {k: round(v/25.4, 1) if v is not None else None for k, v in s.items()}
          for s in stats
      ]

      print(f"Stats for {county}: {rounded_stats}")

['Hawaiʻi' 'Honolulu' 'Kauaʻi' 'Maui']
Processing date: 2026-03-10 cumulative
Stats for Hawaiʻi: [{'min': 0.0, 'max': 0.0, 'mean': 0.0}]
Stats for Honolulu: [{'min': 0.0, 'max': 0.6, 'mean': 0.2}]
Stats for Kauaʻi: [{'min': 0.2, 'max': 6.3, 'mean': 2.0}]
Stats for Maui: [{'min': 0.0, 'max': 0.0, 'mean': 0.0}]
Processing date: 2026-03-11 cumulative
Stats for Hawaiʻi: [{'min': 0.0, 'max': 1.4, 'mean': 0.3}]
Stats for Honolulu: [{'min': 1.0, 'max': 5.4, 'mean': 2.4}]
Stats for Kauaʻi: [{'min': 0.2, 'max': 8.4, 'mean': 3.0}]
Stats for Maui: [{'min': 0.2, 'max': 2.9, 'mean': 1.4}]
Processing date: 2026-03-12 cumulative
Stats for Hawaiʻi: [{'min': 0.1, 'max': 1.9, 'mean': 0.7}]
Stats for Honolulu: [{'min': 1.5, 'max': 8.4, 'mean': 4.3}]
Stats for Kauaʻi: [{'min': 0.7, 'max': 14.0, 'mean': 5.2}]
Stats for Maui: [{'min': 0.6, 'max': 6.2, 'mean': 2.7}]
Processing date: 2026-03-13 cumulative
Stats for Hawaiʻi: [{'min': 0.3, 'max': 10.7, 'mean': 1.8}]
Stats for Honolulu: [{'min': 3.8, 'max': 22.0

In [35]:
gdf = gpd.read_file(shapefile).dissolve(by="county", as_index=False).reset_index(drop=True)

print(gdf["county"].unique())  

for date in np.arange(10, 16+1):
  print(f"Processing date: 2026-03-{date:02d}")
  tif_path = f"./storm_data/2026_03_{date:02d}.tif"

  for county in gdf["county"].unique():
      county_gdf = gdf[gdf["county"] == county]
      stats = zonal_stats(county_gdf, tif_path, stats=["mean", "min", "max"])

      # Round values
      rounded_stats = [
          {
              k: round(v / 25.4, 1) if v is not None else None
              for k, v in s.items()
          }
          for s in stats
      ]

      print(f"Stats for {county}: {rounded_stats}")

['Hawaiʻi' 'Honolulu' 'Kauaʻi' 'Maui']
Processing date: 2026-03-10
Stats for Hawaiʻi: [{'min': 0.0, 'max': 0.0, 'mean': 0.0}]
Stats for Honolulu: [{'min': 0.0, 'max': 0.6, 'mean': 0.2}]
Stats for Kauaʻi: [{'min': 0.2, 'max': 6.3, 'mean': 2.0}]
Stats for Maui: [{'min': 0.0, 'max': 0.0, 'mean': 0.0}]
Processing date: 2026-03-11
Stats for Hawaiʻi: [{'min': 0.0, 'max': 1.4, 'mean': 0.3}]
Stats for Honolulu: [{'min': 0.9, 'max': 4.8, 'mean': 2.2}]
Stats for Kauaʻi: [{'min': 0.0, 'max': 2.7, 'mean': 1.1}]
Stats for Maui: [{'min': 0.2, 'max': 2.9, 'mean': 1.4}]
Processing date: 2026-03-12
Stats for Hawaiʻi: [{'min': 0.0, 'max': 1.0, 'mean': 0.4}]
Stats for Honolulu: [{'min': 0.4, 'max': 4.3, 'mean': 1.9}]
Stats for Kauaʻi: [{'min': 0.4, 'max': 5.6, 'mean': 2.1}]
Stats for Maui: [{'min': 0.2, 'max': 3.9, 'mean': 1.3}]
Processing date: 2026-03-13
Stats for Hawaiʻi: [{'min': 0.0, 'max': 9.9, 'mean': 1.1}]
Stats for Honolulu: [{'min': 2.3, 'max': 13.8, 'mean': 6.8}]
Stats for Kauaʻi: [{'min': 0.8